In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT")
# print(GOOGLE_API_KEY)
# print(LANGCHAIN_PROJECT)

In [8]:
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_PROJECT"] = LANGCHAIN_PROJECT
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [9]:
# print(TAVILY_API_KEY)

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=GOOGLE_API_KEY)
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY)

In [11]:
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# from langchain_groq import ChatGroq
# import os
# llm=ChatGroq(model_name="Gemma2-9b-it")

## Simple AI Assistant

In [12]:
while True:
    question=input("Enter your question. If you want to quit the chat, write quit.")
    if question != "quit":
        print(llm.invoke(question).content)
    else:
        print("Thank you for using the AI assistant!")
        break

I'm sorry, but as an AI, I don't know your name. I don't have access to any personal information about you.

You would need to tell me your name if you'd like me to know it!
As an AI, I don't have feelings or personal experiences, so I can't really be "good" or "bad" in the human sense.

However, I'm operating perfectly and ready to assist you!

Thank you for asking! How are you doing today?
I can't tell you the weather directly because I don't know your specific location! Weather conditions vary greatly depending on where you are.

Could you please tell me your **city and state/country**? Once I have that, I can look up the current conditions and forecast for you.

Alternatively, you can easily check it yourself using a weather app, a quick search on Google for "weather in [your city]", or by just looking outside!
Thank you for using the AI assistant!


In [13]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage


In [14]:
store={}

In [20]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [21]:
config = {"configurable": {"session_id": "firsctchat"}}

In [22]:
model_with_memory = RunnableWithMessageHistory(llm, get_session_history)

In [23]:
model_with_memory.invoke(("Hi, I am Swastika"), config=config).content

"Hi Swastika! It's nice to meet you.\n\nThank you for telling me your name. I'll remember it for our current conversation. Please keep in mind that as an AI, I don't store personal information long-term, so I won't remember it if we have a new conversation later.\n\nHow can I help you today, Swastika?"

In [24]:
model_with_memory.invoke(("Hi, please tell me my name."),config=config).content

'Hello Swastika! Your name is Swastika.\n\nI remember you told me that earlier in this conversation. How can I help you, Swastika?'

In [25]:
store

{'firsctchat': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, please tell me my name.', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm sorry, but I don't know your name. As an AI, I don't have access to personal information about you.\n\nIf you'd like me to refer to you by a specific name during our conversation, you're welcome to tell me!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8fd2-a586-7541-8e07-6ec9e7403481-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 511, 'total_tokens': 520, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 454}}), HumanMessage(content='Hi, I am Swastika', additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Swastika! It's nice to meet you.\n\nThank you for telling me your name. I'll remember it for our

## RAG with LCEL

In [26]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [27]:
loader = DirectoryLoader('../data', glob='./*.txt', loader_cls=TextLoader)
docs = loader.load()

### Creating chunks using Recursive Character Text SPlitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, length_function=len)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

db= Chroma.from_documents(documents=new_docs, embedding=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [28]:
# template=f"""Answer the question based on the context.
# Context: {context}
# Question: {question}
# """

# prompt = PromptTemplate.from_template(template)


template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [29]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [30]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Based on the provided context:

Llama 3 is a model that was released in April 2024.

Here are 3 important points:
1.  Llama 3 was released in April 2024.
2.  There is an 8B parameter version of Llama 3.
3.  Services on a website use a Llama 3 model.


## Agents & Tools

In [38]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [40]:
api_wrapper = WikipediaAPIWrapper()

In [41]:
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [42]:
tool.name

'wikipedia'

In [43]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [45]:
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).
Vector embeddings are mathematical representations of data in a high-dimensional s

In [31]:
from langchain_community.tools import YouTubeSearchTool
ytTool = YouTubeSearchTool() 



In [32]:
ytTool.name

'youtube_search'

In [40]:
ytTool.run("Krish Naik")

"['https://www.youtube.com/watch?v=JxgmHe2NyeY&pp=ygUKS3Jpc2ggTmFpaw%3D%3D', 'https://www.youtube.com/watch?v=LZzq1zSL1bs&pp=ygUKS3Jpc2ggTmFpaw%3D%3D']"

In [41]:
# from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily import TavilySearch

Tavily Crawls the internet

In [37]:
tool3 = TavilySearch()

In [13]:
tool3.invoke("What is the current situation of Israel Iran war?")

{'query': 'What is the current situation of Israel Iran war?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.reuters.com/world/iran/',
   'title': 'Iran War: Latest Breaking News, Updates & Analysis - Reuters',
   'content': 'Real-time Reuters coverage of the Iran war: US-Israel strikes, Iranian retaliation, nuclear threats, oil market shocks, and regional war risks.',
   'score': 0.9845754,
   'raw_content': None},
  {'url': 'https://www.aljazeera.com/tag/israel-iran-conflict/',
   'title': "US-Israel war on Iran | Today's latest from Al Jazeera",
   'content': "Stay on top of US-Israel war on Iran latest developments on the ground with Al Jazeera's fact-based news, exclusive video footage, photos and updated maps.",
   'score': 0.983597,
   'raw_content': None},
  {'url': 'https://www.nbcnews.com/world/iran-war',
   'title': 'Iran War: Latest News, Live Coverage and Video',
   'content': 'The South Asian nation has emerged as an unlik

In [51]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# Requires TAVILY_API_KEY in your .env
tavily_tool = TavilySearch(max_results=3)

agent = create_agent(
    model=llm,
    tools=[tavily_tool],
    system_prompt="You are a helpful assistant. Use Tavily search for up-to-date web info.",
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What are the latest updates in AI this week?"}]}
)
print(response["messages"][-1].content)


This week in AI, there have been several notable updates:

*   **Anthropic's Claude Opus 4.7 Leaks & Full-Stack AI Studio:** Anthropic is reportedly preparing to release Claude Opus 4.7, along with updates such as a full-stack app creation platform, a unified interface for Claude Code, and a beta integration of Claude into Microsoft Word for drafting purposes.
*   **Google I/O Announcements:** Google I/O is expected to unveil significant advancements, including Gemini 3.5, Gemini 4, and DeepSeek 4, which could redefine AI capabilities in natural language processing and search technologies.
*   **Rivian R2's AI Updates:** Rivian's latest AI updates are coming to its newest vehicle, the R2. Future over-the-air updates will enable the AI to navigate stop signs, traffic signals, and turns.
*   **Google's AI-Generated Avatars for YouTube Shorts:** YouTube has introduced a new feature that allows users to create AI avatars with their likeness for use in Shorts. These avatars will include You